In [1]:
# =============================================================================
# Standard Library
# =============================================================================
import re

# =============================================================================
# Data Processing
# =============================================================================
import pandas as pd
import numpy as np
import ast

# =============================================================================
# Lexical Retrieval
# =============================================================================
from rank_bm25 import BM25Okapi

# =============================================================================
# Similarity Search
# =============================================================================
from sklearn.metrics.pairwise import cosine_similarity

# =============================================================================
# Vector Database
# =============================================================================
import chromadb

# =============================================================================
#  CrossEncoder Model
# =============================================================================
from sentence_transformers import CrossEncoder

# =============================================================================
# Embedding Models
# =============================================================================
from sentence_transformers import SentenceTransformer

In [2]:
# ==========================================================
# Load Existing Chroma Database
# ==========================================================
client = chromadb.PersistentClient(path="./chroma_db")
collection = client.get_collection(name="amazon_qa_semantic")

print("Chroma collection loaded successfully.")
print("Total documents:", collection.count())

Chroma collection loaded successfully.
Total documents: 113871


In [3]:
# =============================================================================
# Load Lexical Dataset
# =============================================================================
lexical_dataset = pd.read_csv("lexical_tokens.csv")
print(lexical_dataset.shape)

lexical_dataset.head()

(105096, 9)


,chunk_id,QuestionID,Category,QuestionType,QuestionTime,chunk_index,chunk_text,search_text,lexical_tokens
0,row_0_chunk_0,C15Q2112,Tools and Home Improvement,open-ended,2013-04-20,0,dimensions item,Tools and Home Improvement open-ended dimensio...,"['tools', 'and', 'home', 'improvement', 'open'..."
1,row_1_chunk_0,C9Q4595,Home and Kitchen,open-ended,2014-02-06,0,much booze hold poured booze measuring cup sun...,Home and Kitchen open-ended much booze hold po...,"['home', 'and', 'kitchen', 'open', 'ended', 'm..."
2,row_2_chunk_0,C4Q7999,Cell Phones and Accessories,open-ended,2014-08-09,0,case fit nokia lumia 520 yes fits great great ...,Cell Phones and Accessories open-ended case fi...,"['cell', 'phones', 'and', 'accessories', 'open..."
3,row_3_chunk_0,C8Q8916,Health and Personal Care,open-ended,2014-04-25,0,folded sitting position high ground seat 30 in...,Health and Personal Care open-ended folded sit...,"['health', 'and', 'personal', 'care', 'open', ..."
4,row_4_chunk_0,C14Q905,Sports and Outdoors,open-ended,2015-04-15,0,long leave get max sweat benefit keep thinking...,Sports and Outdoors open-ended long leave get ...,"['sports', 'and', 'outdoors', 'open', 'ended',..."


In [4]:
# =============================================================================
# Restore Token Lists
# =============================================================================
lexical_dataset["lexical_tokens"] = (lexical_dataset["lexical_tokens"].apply(ast.literal_eval))

In [5]:
# =============================================================================
# Build BM25 Index
# =============================================================================
bm25 = BM25Okapi(lexical_dataset["lexical_tokens"].tolist())
print("BM25 Index Created Successfully.")

BM25 Index Created Successfully.


In [6]:
# =============================================================================
# Load Embedding Model
# =============================================================================
embedding_model = SentenceTransformer( "sentence-transformers/all-MiniLM-L6-v2")
print("Embedding Model Loaded.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding Model Loaded.


In [7]:
# =============================================================================
# Semantic Retrieval
# =============================================================================
def semantic_search(query, top_k=5):

    query_embedding = embedding_model.encode( query, normalize_embeddings=True).tolist()

    results = collection.query( query_embeddings=[query_embedding], n_results=top_k,
        include=[
            "documents",
            "metadatas",
            "distances",],)

    return results

In [8]:
# =============================================================================
# BM25 Retrieval
# =============================================================================
def lexical_search(query, top_k=5):

    query_tokens = query.lower().split()
    scores = bm25.get_scores(query_tokens)
    top_indices = scores.argsort()[-top_k:][::-1]
    results = lexical_dataset.iloc[top_indices].copy()
    results["bm25_score"] = scores[top_indices]

    return results

In [9]:
# =============================================================================
# Hybrid Search
# =============================================================================
def hybrid_search( query, semantic_k=20, lexical_k=20):

    semantic_results = semantic_search( query, top_k=semantic_k,)

    lexical_results = lexical_search( query, top_k=lexical_k,)

    return {
        "semantic": semantic_results,
        "lexical": lexical_results,

    }

In [10]:
# =============================================================================
# Test Retrieval
# =============================================================================
query = "Will these mud flaps fit a 2013 F350 dually?"
results = hybrid_search(query)

In [11]:
query = "will they fit 2013 f350 dually"

semantic_results = semantic_search(query)

print(semantic_results["documents"][0])

['Question: what years will it fit Answer: does this fit a ford f350 dually 2005 model like it came from factory', "Question: will these fit a 2013 ford f150-fx4 model with little factory fender flares?? Answer: i believe these will only fit an f150 without fender flares and without the rear wheel well liner (which most f150s don't have). check out weathertech's website or contact weathertech direct to confirm.", 'Question: does this fit on a 2001 f350 superduty 4x4? Answer: yes they will. they fit 99 to 07.', 'Question: will these fit a 2013 ford f150-fx4 model with little factory fender flares?? Answer: i found out the hard way. anybody need some flaps for a 2013 f150 without flares?', "Question: will these fit on a 2004 f250 6.0 with a solid front axle Answer: i doubt they will. a 350 has a larger front wheel bearing. meaning a bigger hub. i'm sure mm has one for it."]


In [12]:
# =============================================================================
# Display Semantic Results
# =============================================================================
for i, doc in enumerate(results["semantic"]["documents"][0]):

    print("=" * 80)
    print("Rank:", i + 1)
    print(doc)
    print()

Rank: 1
Question: will these fit a 2013 ford f150-fx4 model with little factory fender flares?? Answer: i found out the hard way. anybody need some flaps for a 2013 f150 without flares?

Rank: 2
Question: will these work with mud flaps? Answer: yes, i just put them on my 2014 xlt and they fit perfectly and look great. remove the tire first and it is ez. sketch directions are a waste of time until you have them installed then they become obvious.installation on a warm day with the liners in the sun for a time also makes it much easier.

Rank: 3
Question: will these work with mud flaps? Answer: yes

Rank: 4
Question: will these work with mud flaps? Answer: yes! anad they look great. better than without having them in place.

Rank: 5
Question: will these fit a 2013 ford f150-fx4 model with little factory fender flares?? Answer: i believe these will only fit an f150 without fender flares and without the rear wheel well liner (which most f150s don't have). check out weathertech's website or

In [13]:
lexical_results = lexical_search(query)

lexical_results[[
    "QuestionID",
    "chunk_text"
]].head(10)

,QuestionID,chunk_text
29337,C1Q5917,years fit fit ford f350 dually 2005 model like...
28318,C1Q10157,fit 2001 f350 superduty 4x4 yes fit 99 07
97691,C1Q6935,fit 2005 f350 ford super bed truck
60526,C1Q9360,fit 94 ford f350 yes line holes simply installed
55668,C1Q6786,hello fit 1997 f350 auto 4x4 73 diesel 04 model


In [14]:
# =============================================================================
# Display BM25 Results
# =============================================================================
results["lexical"][
    ["chunk_text",
     "bm25_score",]
]

,chunk_text,bm25_score
88403,work mud flaps yes,23.457291
61599,work mud flaps yes anad look great better with...,20.650133
76126,fit 2013 ford f150fx4 model little factory fen...,19.547329
55670,mud flaps front back yes fit dont need jack tu...,19.226969
43685,work mud flaps yes put 2014 xlt fit perfectly ...,16.524178
88760,wide tape going install mud guards 2013 4runne...,16.203839
10195,fit mud runs yes crawling mud,15.967673
28318,fit 2001 f350 superduty 4x4 yes fit 99 07,15.803218
97691,fit 2005 f350 ford super bed truck,15.609753
29337,years fit fit ford f350 dually 2005 model like...,15.234906


In [15]:
# =============================================================================
# Load CrossEncoder Model
# =============================================================================
# CrossEncoder re-ranks retrieved documents by jointly encoding
# the query and each candidate document.

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("CrossEncoder loaded successfully.")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

CrossEncoder loaded successfully.


In [16]:
# =============================================================================
# CrossEncoder Reranking
# =============================================================================

def rerank_results(query, hybrid_results, top_k=10):
    """
    Re-rank retrieved documents using CrossEncoder.
    """

    candidates = []
    
    # ---------------- Semantic Results ----------------
    semantic_docs = hybrid_results["semantic"]["documents"][0]
    semantic_meta = hybrid_results["semantic"]["metadatas"][0]

    for doc, meta in zip(semantic_docs, semantic_meta):

        candidates.append({

            "document": doc,
            "metadata": meta,
            "source": "semantic"})

    # ---------------- Lexical Results ----------------
    for _, row in hybrid_results["lexical"].iterrows():

        candidates.append({

            "document": row["chunk_text"],
            "metadata": {

                "QuestionID": row["QuestionID"],
                "Category": row["Category"],
                "QuestionType": row["QuestionType"],
                "QuestionTime": row["QuestionTime"]},
            "source": "lexical"

        })

    # Create query-document pairs
    pairs = [(query, candidate["document"])

        for candidate in candidates]

    # Predict relevance scores
    scores = reranker.predict(pairs)

    # Attach scores
    for candidate, score in zip(candidates, scores):

        candidate["rerank_score"] = float(score)

    # Sort by score
    candidates = sorted(
        candidates,
        key=lambda x: x["rerank_score"],
        reverse=True)

    return candidates[:top_k]

In [17]:
# =============================================================================
# Remove Duplicate Chunks
# =============================================================================

def remove_duplicates(reranked_results):
    """
    Remove duplicated chunks after reranking.
    """

    unique_chunks = []

    seen = set()

    for item in reranked_results:

        text = item["document"].strip()

        if text not in seen:

            seen.add(text)

            unique_chunks.append(item)

    return unique_chunks

In [18]:
# =============================================================================
# Context Building
# =============================================================================

def build_context(results, max_chunks=5):
    """
    Build the final context passed to the LLM.
    """

    context = ""

    selected_chunks = results[:max_chunks]

    for i, item in enumerate(selected_chunks, start=1):

        context += (
            f"Context {i}\n"
            f"{item['document']}\n\n")

    return context

In [19]:
# =============================================================================
# Retrieve Final Context
# =============================================================================
def retrieve_context(query):

    # -------------------------------
    # Hybrid Retrieval
    # -------------------------------
    hybrid_results = hybrid_search(query)

    # -------------------------------
    # CrossEncoder Reranking
    # -------------------------------
    reranked_results = rerank_results( query, hybrid_results, top_k=10)

    # -------------------------------
    # Remove Duplicate Chunks
    # -------------------------------
    unique_results = remove_duplicates( reranked_results)

    # -------------------------------
    # Build Context
    # -------------------------------
    final_context = build_context( unique_results, max_chunks=8)

    return final_context, unique_results

In [20]:
# =============================================================================
# Test Full Retrieval Pipeline
# =============================================================================
query = "will they fit 2013 f350 dually"
context, retrieved_chunks = retrieve_context(query)

print(context)

Context 1
years fit fit ford f350 dually 2005 model like came factory

Context 2
Question: what years will it fit Answer: does this fit a ford f350 dually 2005 model like it came from factory

Context 3
Question: will they fit a 2014 ford f150 stx with out taking off rear tires Answer: yep. on my model (2013 stx), i had to put them over the fender flare, but doesn't affect performance. look really good installed.

Context 4
Question: does this fit on a 2001 f350 superduty 4x4? Answer: yes they will. they fit 99 to 07.

Context 5
Question: will these fit a 2013 ford f150-fx4 model with little factory fender flares?? Answer: i believe these will only fit an f150 without fender flares and without the rear wheel well liner (which most f150s don't have). check out weathertech's website or contact weathertech direct to confirm.

Context 6
anyone use inflating dually ford f350 attach quickconnect dual chuck yea read kinds reviews bought one think great price good job last used summer air tire

In [21]:
%whos

Variable              Type                   Data/Info
------------------------------------------------------
BM25Okapi             type                   <class 'rank_bm25.BM25Okapi'>
CrossEncoder          ABCMeta                <class 'sentence_transfor<...>oder.model.CrossEncoder'>
SentenceTransformer   ABCMeta                <class 'sentence_transfor<...>del.SentenceTransformer'>
ast                   module                 <module 'ast' from 'C:\\U<...>\envs\\rag\\Lib\\ast.py'>
bm25                  BM25Okapi              <rank_bm25.BM25Okapi obje<...>ct at 0x000002260B35AE90>
build_context         function               <function build_context at 0x0000022631C2A020>
chromadb              module                 <module 'chromadb' from '<...>\\chromadb\\__init__.py'>
client                Client                 <chromadb.api.client.Clie<...>ct at 0x000002260B376FD0>
collection            Collection             Collection(name=amazon_qa_semantic)
context               str           

In [22]:
print(type(context))
print(type(retrieved_chunks))

<class 'str'>
<class 'list'>


In [23]:
type(retrieved_chunks[0])

dict

In [24]:
print(type(retrieved_chunks))

<class 'list'>


In [25]:
# =============================================================================
# Convert Retrieved Chunks to DataFrame
# =============================================================================
retrieved_chunks_df = pd.DataFrame(retrieved_chunks)

print(retrieved_chunks_df.shape)
retrieved_chunks_df.head()

(10, 4)


,document,metadata,source,rerank_score
0,years fit fit ford f350 dually 2005 model like...,"{'QuestionID': 'C1Q5917', 'Category': 'Automot...",lexical,4.619771
1,Question: what years will it fit Answer: does ...,"{'Category': 'Automotive', 'QuestionID': 'C1Q5...",semantic,4.434900
2,Question: will they fit a 2014 ford f150 stx w...,"{'QuestionTime': '2015-01-22', 'QuestionID': '...",semantic,3.867304
3,Question: does this fit on a 2001 f350 superdu...,"{'QuestionID': 'C1Q10157', 'QuestionTime': '20...",semantic,3.701295
4,Question: will these fit a 2013 ford f150-fx4 ...,"{'QuestionType': 'yes/no', 'QuestionID': 'C1Q8...",semantic,3.605453


In [26]:
# =============================================================================
# Save Retrieved Chunks
# =============================================================================
retrieved_chunks_df = pd.DataFrame(retrieved_chunks)
retrieved_chunks_df.to_csv( "retrieved_chunks.csv", index=False)

print("Retrieved chunks saved successfully.")

Retrieved chunks saved successfully.
